# EXA-STAR: neuro-evolved ViT-MAE foundation model on HCP fMRI (Kaggle T4x2)

Evolves a masked-autoencoder vision transformer whose encoder/decoder internals are an EXAMM-style
block graph, trained on the full subject-organized HCP corpus with a subject-level 70/10/20 split.

**Setup expected:**
- The HCP corpus (subject subdirs of `.npz`) mounted as a **Kaggle Dataset** at `HCP_ROOT`.
- The exa-star repo importable (clone it or mount it as a dataset; see the imports cell).

**Kaggle notes:** sessions time out (~12h) and GPU quota is weekly, so the loop **checkpoints to
`/kaggle/working` and resumes** automatically. This first version trains on a **single T4**; two-GPU
genome-parallelism is a later enhancement (see the final markdown cell).


## 1. Configuration

In [2]:
import os

# --- repo + data locations (EDIT these) ---
REPO_PATH = "/kaggle/working/exa-star"                 # where the repo is cloned on Kaggle
REPO_URL = "https://github.com/axj2613/exa-ae.git"     # your fork; add a token here if private
REPO_BRANCH = "autoencoder-aryan"
HCP_ROOT = "/kaggle/working/hcp_complete"          # dir containing the <subject_id>/ subdirs
ATLAS_COORDS = os.path.join(REPO_PATH, "datasets/hcp/atlases/A424_Coordinates.dat")

WORKDIR = "/kaggle/working"
SPLIT_PATH = os.path.join(WORKDIR, "subject_split.json")     # persisted so resume/eval reuse the SAME split
STATS_PATH = os.path.join(WORKDIR, "norm_stats.npz")         # frozen train-split normalization
LENGTH_INDEX_PATH = os.path.join(WORKDIR, "length_index.json")  # cached recording lengths (header-only scan)
CHECKPOINT_PATH = os.path.join(WORKDIR, "evolution_checkpoint.pkl")
BEST_GENOME_PATH = os.path.join(WORKDIR, "best_genome.pkl")
HISTORY_PATH = os.path.join(WORKDIR, "fitness_history.json")   # best/mean fitness per checkpoint
PROGRESS_PLOT_PATH = os.path.join(WORKDIR, "evolution_progress.png")
PRETRAINED_SEED_PATH = os.path.join(WORKDIR, "pretrained_seed.pkl")  # from pretrain_seed_kaggle.ipynb; loaded as the search seed

# --- windowing ---
# A handful of HCP runs are truncated (some tasks are as short as ~35 timepoints), so no single
# window can include literally every recording without being uselessly tiny. window=120 (6 temporal
# patches) includes 99.7% of recordings -- all complete runs incl. EMOTION -- and the dataset logs
# how many truncated recordings (<window) it excludes. Raise toward 140/160 for more temporal
# context at the cost of dropping a few more short runs.
WINDOW_LENGTH = 20          # == TIME_PATCH_SIZE => ONE temporal patch (424 tokens). Multiple 20-TR patches are temporally decorrelated and only dilute the spatial (FC) signal, stalling learning; one patch trains cleanly toward the ~0.22 R2 ceiling
TIME_PATCH_SIZE = 20        # window==patch => 1 temporal patch; token = one parcel's 20-TR window
MASK_RATIO = 0.5            # BrainLM's ratio; more visible parcels => stronger reconstruction signal
SPLIT_RATIOS = (0.7, 0.1, 0.2)

# --- model / evolution (full-scale search config) ---
D_MODEL = 128                # baked into the topology -- this is your final model width
NUM_HEADS = 4
D_FF = 512                   # 4x d_model, standard FFN size inside each attention block
DROPOUT = 0.1
USE_PRETRAINED_SEED = True   # True: seed the search from pretrained_seed.pkl (recommended). False:
                             # build a fresh UNtrained seed of the depths below -- the "grow from
                             # scratch" experiment: does the search dodge the 0.98 plateau on its own?
SEED_ENCODER_DEPTH = 1       # used ONLY when USE_PRETRAINED_SEED=False (a pretrained seed carries its
SEED_DECODER_DEPTH = 1       # own architecture). Set BOTH to 1 for a minimal 1-encoder/1-decoder start.
# this run is the ATTENTION-ONLY arm; set the full list
# ["attention", "simple", "sequence_lstm", "temporal_lstm"] for the mixed-cell comparison.
NODE_TYPES = ["attention"]
POPULATION_SIZE = 16         # larger pool hedges the noisy proxy + gives the re-rank more survivors
NUM_GENERATIONS = 500        # monitor evolution_progress.png; stop early once it plateaus

# --- per-genome training budget (search only; the winner is trained long separately via
#     train_final_model.py) ---
NUM_ITERATIONS = 4            # 6x60 = 360 steps/genome, up from 4x40=160 (see BATCHES_PER_ITERATION)
BATCHES_PER_ITERATION = 30   # more gradient signal per genome + more Lamarckian accumulation/gen, after the last run plateaued at MSE 0.981
BATCH_SIZE = 128             # 424 tokens now (single patch) => bigger batch fits + less noisy fitness; was 8 
FITNESS_BATCHES = 32         # doubled: fitness was over only 128 windows (very noisy ranking, fidelity rho~0); more val batches = less selection noise
LEARNING_RATE = 0.001
USE_AMP = True               # CUDA mixed precision (tensor cores on T4)

CHECKPOINT_EVERY = 5         # genomes between checkpoints

# --- Progressive Dynamic Hurdles (dynamic per-genome budget; So et al., "The Evolved Transformer") ---
# Escalate training only for genomes that clear the population's mean fitness, so compute
# concentrates on promising lineages and a slightly-better deeper genome earns the steps to
# prove itself (the fix for the depth-growth stall). A cheap fixed proxy budget ranked our
# genomes too noisily (fidelity rho ~0.3) -- PDH is the principled alternative.
USE_PDH = True               # False falls back to the fixed NUM_ITERATIONS x BATCHES_PER_ITERATION budget
PDH_STEP_INCREMENTS = [60, 120, 240]   # gradient steps per stage; stage 0 always runs (cheap screen),
                                       # hurdle-clearers earn +120 then +240 (up to 420 steps total)
PDH_MODELS_PER_HURDLE = 16   # genomes between new hurdles (creates len(PDH_STEP_INCREMENTS)-1 = 2 hurdles)


## 2. Fetch the repo + imports

Clones the repo on a fresh session (or pulls the latest on re-run) so the code always matches
this notebook -- no more manual dataset re-uploads. If your fork is **private**, put a GitHub
token in `REPO_URL` (e.g. from Kaggle Secrets): `https://{token}@github.com/axj2613/exa-ae.git`.

*(If you `git pull` new code into an already-imported session, restart the kernel so Python
reloads the modules.)*

In [3]:
import os
import subprocess
import sys

if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull", "origin", REPO_BRANCH], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import pickle
# expandable segments reduce caching-allocator fragmentation across many differently-sized
# genomes (must be set before torch initializes CUDA -- a fresh kernel picks it up).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


From https://github.com/axj2613/exa-ae
 * branch            autoencoder-aryan -> FETCH_HEAD


Already up to date.
device: cuda Tesla T4


## 3. Dataset

Builds the subject-level split (persisted) and freezes per-parcel normalization from the training
split (persisted). Both files are reused on resume and by the downstream embedding extraction so
the split and normalization are identical everywhere.

In [4]:
from time_series.hcp_window_dataset import HCPWindowDataset

dataset = HCPWindowDataset(
    root_dir=HCP_ROOT,
    atlas_coordinates_filename=ATLAS_COORDS,
    window_length=WINDOW_LENGTH,
    split_ratios=SPLIT_RATIOS,
    split_path=SPLIT_PATH,
    stats_path=STATS_PATH,
    length_index_path=LENGTH_INDEX_PATH,
)
print("parcels:", dataset.num_parcels)
print("split sizes:", {k: len(v) for k, v in dataset.splits.items()})


2026-08-09 05:59:14.542 | INFO     | time_series.hcp_window_dataset:__init__:110 - discovered 1098 subjects, 19134 recordings under '/kaggle/working/hcp_complete'
2026-08-09 05:59:14.551 | INFO     | time_series.hcp_window_dataset:_build_length_index:198 - loaded recording length index from '/kaggle/working/length_index.json'
2026-08-09 05:59:14.557 | INFO     | time_series.hcp_window_dataset:_make_or_load_split:153 - loaded subject split from '/kaggle/working/subject_split.json'
2026-08-09 05:59:14.681 | INFO     | time_series.hcp_window_dataset:_load_or_compute_stats:293 - loaded normalization stats from '/kaggle/working/norm_stats.npz'


parcels: 424
split sizes: {'train': 769, 'val': 110, 'test': 219}


## 4. Build or resume the population

If a checkpoint exists (from a previous timed-out session) it is loaded and the run continues;
otherwise a fresh seed genome + population is created.

In [5]:
from population.single_population import SinglePopulation
from evolution.vision_transformer_block_edge_generator import VisionTransformerBlockEdgeGenerator
from evolution.vision_transformer_block_node_generator import VisionTransformerBlockNodeGenerator
from evolution.vision_transformer_block_reproduction_selector import VisionTransformerBlockReproductionSelector
from genomes.vision_transformer_block_genome import VisionTransformerBlockGenome
from weight_generators.lamarckian_block_weight_generator import LamarckianBlockWeightGenerator
from evolution.checkpoint import save_checkpoint, load_checkpoint

if os.path.exists(CHECKPOINT_PATH):
    state = load_checkpoint(CHECKPOINT_PATH)
    population = state["population_strategy"]
    start_generation = state["generation"]
    resumed_pdh_state = state.get("pdh_state")   # PDH hurdle schedule (None for old checkpoints)
    # the checkpoint was pickled on CPU; re-home the whole population (and seed) to the GPU so
    # it isn't device-mixed with the GPU children generated after resuming.
    for genome in population.population:
        genome.to(device)
    if population.seed_genome is not None:
        population.seed_genome.to(device)
    print(f"resumed from checkpoint at generation {start_generation}")
else:
    weight_generator = LamarckianBlockWeightGenerator()
    node_generator = VisionTransformerBlockNodeGenerator(
        num_heads=NUM_HEADS, d_ff=D_FF, dropout=DROPOUT, allowed_node_types=NODE_TYPES
    )
    edge_generator = VisionTransformerBlockEdgeGenerator()

    if USE_PRETRAINED_SEED and os.path.exists(PRETRAINED_SEED_PATH):
        # start the search from the PRE-TRAINED base (pretrain_seed_kaggle.ipynb) so every genome
        # inherits a working reconstruction model via Lamarckian inheritance, rather than a shallow
        # untrained seed the search can't grow into (the ~MSE 0.99 stall).
        with open(PRETRAINED_SEED_PATH, "rb") as seed_file:
            seed_genome = pickle.load(seed_file)
        assert seed_genome.window_length == WINDOW_LENGTH, "pretrained seed window_length != WINDOW_LENGTH"
        seed_genome.generation_number = 0
        rep = seed_genome.parameter_report()
        print(f"seeding evolution from PRE-TRAINED seed {PRETRAINED_SEED_PATH} "
              f"({rep['total_active_parameters']:,} params, {rep['node_type_counts_by_region']})")
    else:
        seed_genome = VisionTransformerBlockGenome(
            generation_number=0, num_parcels=dataset.num_parcels, window_length=WINDOW_LENGTH,
            parcel_coordinates=dataset.parcel_coordinates, d_model=D_MODEL, num_heads=NUM_HEADS,
            d_ff=D_FF, dropout=DROPOUT, time_patch_size=TIME_PATCH_SIZE, mask_ratio=MASK_RATIO,
            encoder_depth=SEED_ENCODER_DEPTH, decoder_depth=SEED_DECODER_DEPTH,
            weight_generator=weight_generator,
        )
        rep = seed_genome.parameter_report()
        if USE_PRETRAINED_SEED:
            print(f"WARNING: USE_PRETRAINED_SEED=True but no seed at {PRETRAINED_SEED_PATH} -- evolving from an "
                  f"UNTRAINED seed. Run pretrain_seed_kaggle.ipynb first for best results.")
        else:
            print(f"USE_PRETRAINED_SEED=False -- grow-from-scratch experiment: evolving from a fresh untrained "
                  f"{SEED_ENCODER_DEPTH}-enc/{SEED_DECODER_DEPTH}-dec seed "
                  f"({rep['total_active_parameters']:,} params, {rep['node_type_counts_by_region']})")

    population = SinglePopulation(
        population_size=POPULATION_SIZE, seed_genome=seed_genome,
        reproduction_selector=VisionTransformerBlockReproductionSelector(
            node_generator=node_generator, edge_generator=edge_generator, weight_generator=weight_generator,
        ),
    )
    start_generation = 0
    resumed_pdh_state = None
    print("starting fresh evolution with node types:", NODE_TYPES)


resumed from checkpoint at generation 426


## 5. Evolution loop (multi-GPU)

`resolve_devices()` finds every GPU (both T4s on Kaggle's T4x2), and `evolve_parallel` generates
one genome per device and **trains them concurrently, one per GPU** -- ~2x throughput. Each genome
trains on the **train** split (mixed precision) and is scored on the **validation** split.
Checkpoints every ~`CHECKPOINT_EVERY` genomes survive session restarts.

*(The reproduction operators print verbosely; filter stdout/loguru if you want a quieter log.)*

In [6]:
from evolution.parallel_training import evolve_parallel, resolve_devices
from evolution.fitness_history import FitnessHistory
from evolution.progressive_hurdles import ProgressiveDynamicHurdles

config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, tuple, list))}
devices = resolve_devices()
print("training devices:", devices)
history = FitnessHistory(HISTORY_PATH)   # reloads prior history on resume, so the curve is continuous
train_kwargs = dict(
    iterations=NUM_ITERATIONS, batches_per_iteration=BATCHES_PER_ITERATION,
    batch_size=BATCH_SIZE, fitness_batches=FITNESS_BATCHES, use_amp=USE_AMP,
)

# Progressive Dynamic Hurdles: hurdle-escalated per-genome budget (see config cell). With PDH on,
# iterations/batches_per_iteration in train_kwargs are ignored -- PDH supplies the step schedule.
pdh = None
if USE_PDH:
    pdh = ProgressiveDynamicHurdles(PDH_STEP_INCREMENTS, PDH_MODELS_PER_HURDLE)
    if resumed_pdh_state is not None:
        pdh.load_state_dict(resumed_pdh_state)
        print(f"[PDH] resumed with {len(pdh.hurdles)} hurdle(s): {[round(h, 5) for h in pdh.hurdles]}")
    print(f"[PDH] enabled: stages {PDH_STEP_INCREMENTS} steps, new hurdle every {PDH_MODELS_PER_HURDLE} genomes")

generation = start_generation
last_checkpoint = start_generation
progress = tqdm(total=NUM_GENERATIONS, initial=start_generation)
while generation < NUM_GENERATIONS:
    k = min(len(devices), NUM_GENERATIONS - generation)
    evolve_parallel(population, dataset, devices[:k], LEARNING_RATE, num_genomes=k, pdh=pdh, **train_kwargs)
    generation += k
    progress.update(k)
    if generation - last_checkpoint >= CHECKPOINT_EVERY:
        # save_checkpoint moves genomes to CPU to pickle, then restores each to its own device.
        save_checkpoint(CHECKPOINT_PATH, population, generation, config=config,
                        pdh_state=pdh.state_dict() if pdh is not None else None)
        history.record(generation, population)   # persisted best/mean fitness for the progress plot
        best = population.population[0]
        with open(BEST_GENOME_PATH, "wb") as best_file:
            pickle.dump(best, best_file)
        print(f"[checkpoint @ gen {generation}] best validation MSE: {best.fitness:.6f}")
        last_checkpoint = generation
progress.close()


training devices: [device(type='cuda', index=0), device(type='cuda', index=1)]
[PDH] resumed with 2 hurdle(s): [0.84675, 0.78407]
[PDH] enabled: stages [60, 120, 240] steps, new hurdle every 16 genomes


 85%|########5 | 426/500 [00:00<?, ?it/s]

REPRODUCTION METHOD: DisableEdge
REPRODUCTION METHOD: DisableEdge
iteration 0 train loss: 0.718293
iteration 0 train loss: 0.731762
final fitness (validation MSE): 0.725941 | active params: 2,987,665 (2,974,141 evolved) | active hidden nodes: 15, edges: 61 | types: {'AttentionBlockNode': 15} (encoder: {'AttentionBlockNode': 11}, decoder: {'AttentionBlockNode': 4})
final fitness (validation MSE): 0.727586 | active params: 2,591,113 (2,577,589 evolved) | active hidden nodes: 13, edges: 53 | types: {'AttentionBlockNode': 13} (encoder: {'AttentionBlockNode': 9}, decoder: {'AttentionBlockNode': 4})
iteration 0 train loss: 0.722400
iteration 0 train loss: 0.732643
final fitness (validation MSE): 0.729293 | active params: 2,987,665 (2,974,141 evolved) | active hidden nodes: 15, edges: 61 | types: {'AttentionBlockNode': 15} (encoder: {'AttentionBlockNode': 11}, decoder: {'AttentionBlockNode': 4})
final fitness (validation MSE): 0.726958 | active params: 2,591,113 (2,577,589 evolved) | active h

## 6. Final save + download

All outputs are written to `/kaggle/working` (`WORKDIR`). **`/kaggle/working` is wiped when an
interactive session ends unless you persist it** -- so to keep the trained model do ONE of:
- enable **Persistence -> Files only** in the notebook settings (right sidebar) -- also required
  for the checkpoint/resume across sessions to work; or
- **Save Version** (Save & Run All), which stores `/kaggle/working` as the version's Output; or
- download the files now via the links printed below.

Downstream clinical evaluation (`subject_embeddings.py` -> `eval_cls.py`) needs THREE of these,
not just the genome: `best_genome.pkl`, `subject_split.json`, and `norm_stats.npz` -- so that the
held-out test subjects and the normalization match what the model was trained with.

In [7]:
from IPython.display import FileLink, display

save_checkpoint(CHECKPOINT_PATH, population, generation, config=config)
history.record(generation, population)
history.plot(PROGRESS_PLOT_PATH)   # best/mean validation MSE vs generation -- watch it plateau
best = population.population[0]
with open(BEST_GENOME_PATH, "wb") as best_file:
    pickle.dump(best, best_file)

print("best genome validation MSE:", best.fitness)
report = best.parameter_report()
print(f"best genome size: {report['total_active_parameters']:,} active trainable params "
      f"({report['evolved_active_parameters']:,} evolved) -- vs BrainLM's 111M / 650M")
print(f"  active hidden blocks: {report['num_active_hidden_nodes']} {report['node_type_counts']}, "
      f"active edges: {report['num_active_edges']}")
print(best)

print("\nartifacts in", WORKDIR, "(download these -- the first three are needed for clinical eval):")
for path in [BEST_GENOME_PATH, SPLIT_PATH, STATS_PATH, CHECKPOINT_PATH, LENGTH_INDEX_PATH,
             HISTORY_PATH, PROGRESS_PLOT_PATH]:
    if os.path.exists(path):
        print(f"  {os.path.getsize(path) / 1e6:8.2f} MB  {path}")
        display(FileLink(os.path.relpath(path, WORKDIR)))


saved evolution-progress plot to /kaggle/working/evolution_progress.png (best MSE 0.72211 at generation 500)
best genome validation MSE: 0.7221050411462784
best genome size: 3,582,495 active trainable params (3,568,971 evolved) -- vs BrainLM's 111M / 650M
  active hidden blocks: 18 {'AttentionBlockNode': 18}, active edges: 75
Genome 490 : fitness: 0.7221050411462784
<class 'genomes.vision_transformer_block_genome.VisionTransformerBlockGenome'>
	[block node BlockInputNode, innovation: 0, depth: 0.0, d_model: 128, disabled: False]
		BlockEdge 2 from node 0 to node 1, weight: [tensor(-0.1726, device='cuda:1', requires_grad=True)]
		BlockEdge 87 from node 0 to node 84, weight: [tensor(-0.1195, device='cuda:1', requires_grad=True)]
		BlockEdge 208 from node 0 to node 84, weight: [tensor(0.2725, device='cuda:1', requires_grad=True)]
		BlockEdge 249 from node 0 to node 247, weight: [tensor(0.0905, device='cuda:1', requires_grad=True)]
		BlockEdge 251 from node 0 to node 247, weight: [tensor(-

/kaggle/working/best_genome.pkl

      0.02 MB  /kaggle/working/subject_split.json


/kaggle/working/subject_split.json

      0.00 MB  /kaggle/working/norm_stats.npz


...

/kaggle/working/length_index.json

      0.01 MB  /kaggle/working/fitness_history.json


/kaggle/working/fitness_history.json

      0.06 MB  /kaggle/working/evolution_progress.png


/kaggle/working/evolution_progress.png

## Notes

- **Multi-GPU** is active: `evolve_parallel` trains one genome per detected GPU concurrently
  (generational batching -- generate K, train across `cuda:0`/`cuda:1`, insert K). It falls back to
  a single device (or CPU) automatically. Set `POPULATION_SIZE` >= number of GPUs.
- **Per-genome budget vs. dataset size:** training cost is fixed by `NUM_ITERATIONS x
  BATCHES_PER_ITERATION x BATCH_SIZE`, not by how much data exists -- each genome sees a small
  random slice. To exploit more data / get a less noisy fitness signal, raise those budgets (and/or
  `NUM_GENERATIONS`). The proxy-vs-full fidelity experiment
  (`evaluation_scripts/proxy_fidelity_experiment.py`) checks whether the cheap fitness ranks
  architectures the way full training would.